# Gaia XP regression workbook

This notebook is a workbook, not a finished analysis. The code cells marked
YOUR TURN are empty on purpose: you write them. Each one states a goal, a few
hints, and a checkpoint so you know your code worked before moving on. The
point is to build the next phase yourself and own it.

The five notes from the meeting with Josh, verbatim, and where they live here:

1. try the prior other than N(0,1): Glorot, shrink the typical set, fan in
   fan out prior, 1/T .............................................. section 3
2. try regression instead: Laroche, Speagle Gaia ......... sections 1 and 2
3. package GaiaXpy ................................................ section 7
4. how to visualize the uncertainty and compress the samples ...... section 5
5. visuals: each layer and correlation ............................ section 6

Working habits: do the sections in order, they build on each other. When you
are stuck, the exp6 code in `experiments/` has the same patterns on MNIST
(`exp6_sample_metropolis.py`, `exp6_prior_start.py`, `exp6_figures.py`). Read
it for the pattern, then write your own version here. Copy-pasting defeats
the purpose.

## 0. Setup

Three packages are missing from the venv: `h5py` reads the Zenodo table,
`pandas` is what GaiaXpy takes as input, and `GaiaXPy` turns XP coefficients
back into spectra you can look at (it needs Python 3.10 or newer, the venv
is 3.12). Run the install once, then restart the kernel.

In [4]:
# run once, then restart the kernel
%pip install h5py pandas GaiaXPy

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd() if (Path.cwd() / "experiments").exists() else Path.cwd().parent
DATA = ROOT / "data" / "gaia_xp"
DATA.mkdir(parents=True, exist_ok=True)
DEV = "mps" if torch.backends.mps.is_available() else "cpu"
RNG = np.random.default_rng(0)
print(f"root {ROOT}\ndevice {DEV}")

root /Users/yasiralsugair/UofT/empirical
device mps


### 0.1 Get the data

The dataset is the catalogue behind Laroche and Speagle 2025 (ApJ 979, 5,
[arXiv:2404.07316](https://arxiv.org/abs/2404.07316)), Zenodo record
[14041773](https://zenodo.org/records/14041773). Josh: you should just need
the single main table. That is `xp_apogee_cat.h5`, 1.6 GB, described as
"all Gaia XP data and APOGEE stellar labels used in this work". The record
holds eight more files (wavelength-space spectra, covariances for a test
subset, model reconstructions); ignore them until something here needs one.

The cell below downloads the table if it is missing and verifies the
checksum. 1.6 GB, so give it a few minutes.

In [24]:
import hashlib
import urllib.request

MAIN = DATA / "xp_apogee_cat.h5"
URL = "https://zenodo.org/records/14041773/files/xp_apogee_cat.h5?download=1"
MD5 = "5eada07bea841629c4da1d1e0dcb4258"

if not MAIN.exists():
    done = [0]
    def hook(blocks, bs, total):
        got = blocks * bs
        if got - done[0] > 100e6:
            done[0] = got
            print(f"  {got / 1e9:.1f} / {total / 1e9:.1f} GB")
    print(f"downloading {URL}")
    urllib.request.urlretrieve(URL, MAIN, reporthook=hook)

h = hashlib.md5()
with open(MAIN, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
ok = h.hexdigest() == MD5
print(f"{MAIN.name}: {MAIN.stat().st_size / 1e9:.2f} GB, md5 {'ok' if ok else 'MISMATCH, redownload'}")

xp_apogee_cat.h5: 1.60 GB, md5 ok


### 0.2 Look inside the file

YOUR TURN. Never model a file you have not looked at. Open the table with
h5py and walk its contents: every dataset name, shape, and dtype.

Hints: `h5py.File(MAIN)` behaves like a dict, `.keys()` lists the top level,
and `f.visititems(lambda name, obj: print(name, getattr(obj, "shape", "")))`
walks everything in one line.

Checkpoint: you can say how many stars the table holds (the paper's full
APOGEE x XP crossmatch is 502,311), and you can point at the XP coefficient
array (110 per star, 55 BP + 55 RP) and at the APOGEE labels with their
uncertainties.

In [28]:
# YOUR TURN: open MAIN with h5py and print every dataset name, shape, dtype
data = h5py.File(MAIN)
data.visititems(lambda name, obj: print(name, getattr(obj, "shape", "")))

__astropy_table__ (502311,)


In [29]:
from astropy.table import Table
t = Table.read(MAIN, path="__astropy_table__")
t[:5]

coeff_errs,coeffs,ids,phot_g_mean_flux,phot_g_mean_flux_error,GAIAEDR3_SOURCE_ID,FILE,APOGEE_ID,TARGET_ID,APSTAR_ID,ASPCAP_ID,TELESCOPE,LOCATION_ID,FIELD,ALT_ID,RA,DEC,GLON,GLAT,J,J_ERR,H,H_ERR,K,K_ERR,SRC_H,WASH_M,WASH_M_ERR,WASH_T2,WASH_T2_ERR,DDO51,DDO51_ERR,IRAC_3_6,IRAC_3_6_ERR,IRAC_4_5,IRAC_4_5_ERR,IRAC_5_8,IRAC_5_8_ERR,IRAC_8_0,IRAC_8_0_ERR,WISE_4_5,WISE_4_5_ERR,TARG_4_5,TARG_4_5_ERR,WASH_DDO51_GIANT_FLAG,WASH_DDO51_STAR_FLAG,TARG_PMRA,TARG_PMDEC,TARG_PM_SRC,AK_TARG,AK_TARG_METHOD,AK_WISE,SFD_EBV,APOGEE_TARGET1,APOGEE_TARGET2,APOGEE2_TARGET1,APOGEE2_TARGET2,APOGEE2_TARGET3,APOGEE2_TARGET4,TARGFLAGS,SURVEY,PROGRAMNAME,NVISITS,SNR,SNREV,STARFLAG,STARFLAGS,ANDFLAG,ANDFLAGS,VHELIO_AVG,VSCATTER,VERR,RV_TEFF,RV_LOGG,RV_FEH,RV_ALPHA,RV_CARB,RV_CHI2,RV_CCFWHM,RV_AUTOFWHM,RV_FLAG,N_COMPONENTS,MEANFIB,SIGFIB,MIN_H,MAX_H,MIN_JK,MAX_JK,GAIAEDR3_PARALLAX,GAIAEDR3_PARALLAX_ERROR,GAIAEDR3_PMRA,GAIAEDR3_PMRA_ERROR,GAIAEDR3_PMDEC,GAIAEDR3_PMDEC_ERROR,GAIAEDR3_PHOT_G_MEAN_MAG,GAIAEDR3_PHOT_BP_MEAN_MAG,GAIAEDR3_PHOT_RP_MEAN_MAG,GAIAEDR3_DR2_RADIAL_VELOCITY,GAIAEDR3_DR2_RADIAL_VELOCITY_ERROR,GAIAEDR3_R_MED_GEO,GAIAEDR3_R_LO_GEO,GAIAEDR3_R_HI_GEO,GAIAEDR3_R_MED_PHOTOGEO,GAIAEDR3_R_LO_PHOTOGEO,GAIAEDR3_R_HI_PHOTOGEO,ASPCAP_GRID,FPARAM_GRID,CHI2_GRID,FPARAM,FPARAM_COV,ASPCAP_CHI2,PARAM,PARAM_COV,PARAMFLAG,ASPCAPFLAG,ASPCAPFLAGS,FRAC_BADPIX,FRAC_LOWSNR,FRAC_SIGSKY,FELEM,FELEM_ERR,X_H,X_H_ERR,X_M,X_M_ERR,ELEM_CHI2,ELEMFRAC,ELEMFLAG,EXTRATARG,MEMBERFLAG,MEMBER,X_H_SPEC,X_M_SPEC,TEFF,TEFF_ERR,LOGG,LOGG_ERR,M_H,M_H_ERR,ALPHA_M,ALPHA_M_ERR,VMICRO,VMACRO,VSINI,TEFF_SPEC,LOGG_SPEC,C_FE,C_FE_SPEC,C_FE_ERR,C_FE_FLAG,CI_FE,CI_FE_SPEC,CI_FE_ERR,CI_FE_FLAG,N_FE,N_FE_SPEC,N_FE_ERR,N_FE_FLAG,O_FE,O_FE_SPEC,O_FE_ERR,O_FE_FLAG,NA_FE,NA_FE_SPEC,NA_FE_ERR,NA_FE_FLAG,MG_FE,MG_FE_SPEC,MG_FE_ERR,MG_FE_FLAG,AL_FE,AL_FE_SPEC,AL_FE_ERR,AL_FE_FLAG,SI_FE,SI_FE_SPEC,SI_FE_ERR,SI_FE_FLAG,P_FE,P_FE_SPEC,P_FE_ERR,P_FE_FLAG,S_FE,S_FE_SPEC,S_FE_ERR,S_FE_FLAG,K_FE,K_FE_SPEC,K_FE_ERR,K_FE_FLAG,CA_FE,CA_FE_SPEC,CA_FE_ERR,CA_FE_FLAG,TI_FE,TI_FE_SPEC,TI_FE_ERR,TI_FE_FLAG,TIII_FE,TIII_FE_SPEC,TIII_FE_ERR,TIII_FE_FLAG,V_FE,V_FE_SPEC,V_FE_ERR,V_FE_FLAG,CR_FE,CR_FE_SPEC,CR_FE_ERR,CR_FE_FLAG,MN_FE,MN_FE_SPEC,MN_FE_ERR,MN_FE_FLAG,FE_H,FE_H_SPEC,FE_H_ERR,FE_H_FLAG,CO_FE,CO_FE_SPEC,CO_FE_ERR,CO_FE_FLAG,NI_FE,NI_FE_SPEC,NI_FE_ERR,NI_FE_FLAG,CU_FE,CU_FE_SPEC,CU_FE_ERR,CU_FE_FLAG,CE_FE,CE_FE_SPEC,CE_FE_ERR,CE_FE_FLAG,YB_FE,YB_FE_SPEC,YB_FE_ERR,YB_FE_FLAG,VISIT_PK,dist,dist_model_error,dist_error,weighted_dist,weighted_dist_error,age,age_linear_correct,age_lowess_correct,age_total_error,age_model_error,galr,galphi,galz,galr_err,galphi_err,galz_err,galvr,galvt,galvz,galvr_err,galvt_err,galvz_err,galvr_galvt_corr,galvr_galvz_corr,galvt_galvz_corr,e,e_err,zmax,zmax_err,rperi,rperi_err,rap,rap_err,e_zmax_corr,e_rperi_corr,e_rap_corr,zmax_rperi_corr,zmax_rap_corr,rperi_rap_corr,jr,jr_err,Lz,Lz_err,jz,jz_err,jr_Lz_corr,jr_jz_corr,lz_jz_corr,omega_r,omega_r_err,omega_phi,omega_phi_err,omega_z,omega_z_err,theta_r,theta_r_err,theta_phi,theta_phi_err,theta_z,theta_z_err,rl,rl_err,Energy,Energy_err,EminusEc,EminusEc_err
float32[110],float32[110],int64,float32,float32,int64,bytes64,bytes30,bytes58,bytes71,bytes77,bytes6,int32,bytes20,bytes30,float64,float64,float64,float64,float32,float32,float32,float32,float32,float32,bytes16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,int32,float32,float32,bytes16,float32,bytes32,float32,float32,int32,int32,int32,int32,int32,int32,bytes132,bytes32,bytes32,int32,float32,float32,int64,bytes132,int64,bytes132,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,bytes8,"float32[21,9]",float32[21],float32[9],"float32[9,9]",float32,float32[9]

In [30]:
len(t.colnames)

305

In [33]:
print([(c, t[c].shape) for c in t.colnames if t[c].ndim > 1])

[('coeff_errs', (502311, 110)), ('coeffs', (502311, 110)), ('FPARAM_GRID', (502311, 21, 9)), ('CHI2_GRID', (502311, 21)), ('FPARAM', (502311, 9)), ('FPARAM_COV', (502311, 9, 9)), ('PARAM', (502311, 9)), ('PARAM_COV', (502311, 9, 9)), ('PARAMFLAG', (502311, 9)), ('FELEM', (502311, 27)), ('FELEM_ERR', (502311, 27)), ('X_H', (502311, 27)), ('X_H_ERR', (502311, 27)), ('X_M', (502311, 27)), ('X_M_ERR', (502311, 27)), ('ELEM_CHI2', (502311, 27)), ('ELEMFRAC', (502311, 27)), ('ELEMFLAG', (502311, 27)), ('X_H_SPEC', (502311, 27)), ('X_M_SPEC', (502311, 27)), ('VISIT_PK', (502311, 100))]


In [ ]:

[c for c in t.colnames if "ALPHA" in c.upper() or c.upper() in ("TEFF", "LOGG", "M_H")]

['RV_ALPHA', 'TEFF', 'LOGG', 'M_H', 'ALPHA_M', 'ALPHA_M_ERR']

### 0.3 Build the arrays

YOUR TURN. Pull out what the rest of the notebook needs, as float32 numpy
arrays:

    X      (N, 110)  column "coeffs" (raw flux units; "coeff_errs" sits alongside)
    y      (N,)      "ALPHA_M"
    y_err  (N,)      "ALPHA_M_ERR"
    extras (N, 3)    TEFF, LOGG, M_H via column_stack (write the order down)

Conversion rule: numpy takes exactly one astropy Column at a time. Anything
that returns a Table (indexing with several names) must be taken apart
column by column before numpy touches it.

The quality cut is the paper's exact recipe (Section 2.2), so there is
nothing to invent. One correction to earlier drafts of this cell: the paper
imposes NO Teff or logg cut anywhere; the pristine sample is defined purely
by label precision, and cool giants emerge from that. The labels in this
table are the astroNN DR17 ones the paper uses. Build one boolean mask,
conditions chained with `&`, with a labeled `mask.sum()` print after each
stage so your funnel is comparable to the paper's
502,311 to 202,970 (good) to 123,804 (pristine):

1. finiteness: `np.isfinite` on y, y_err, and the extras rows; `y_err > 0`
2. good labels: Teff / sigma(Teff) > 30, sigma(logg) < 0.4,
   sigma([M/H]) < 0.2, 0 < BP-RP < 4, 6 < G < 17.5, STARFLAG == 0,
   ASPCAPFLAG bits 19 and 23 clear
3. pristine: tighten the three label cuts to > 100, < 0.1, < 0.05

Bit test idiom: `(flag & (1 << 19)) == 0` means bit 19 is clear. Hunt the
remaining column names with the colnames filter (patterns ERR, FLAG, MAG,
BP_RP). Landing near 123,804 means your sample is the paper's pristine
sample; record that in a markdown note with the criteria and the seed.

Structure: conversion, mask, and cut live in ONE cell that rebuilds from
`t` at the top, so any rerun starts clean.

Split: one permutation of row numbers, called once.
`idx = RNG.permutation(len(X))`, cut at `int(0.8 * len(X))`, and index
every array with the same two halves. Never call permutation on the data
arrays themselves: each call draws a different order and silently unglues
spectra from labels.

Standardize X, and only X (y and y_err stay in dex, the likelihood needs
them in physical units):

    norm_mu = train_X.mean(axis=0)
    norm_sd = train_X.std(axis=0)
    train_Xs = (train_X - norm_mu) / norm_sd
    test_Xs  = (test_X  - norm_mu) / norm_sd

Keep `norm_mu` and `norm_sd` untouched from here on; sections 4 and 7 need
to undo this.

Checkpoint, one cell you rerun after any mask change: train and test counts
sum to n with zero overlap between the index sets; zero nan or
nonpositive-error rows; y range near -0.1 to +0.4 with median y_err far
below the y spread; train_Xs means at the 1e-5 level and stds at 1.0;
test_Xs stds within a couple percent of 1.0. A test std far below 1 means a
few outlier stars own that column's variance (it happened on the first
pass: one star sat 358 sigma out on coefficient 41); the flag, color, and
magnitude cuts are what remove such stars, so re-check after cutting.

In [90]:
# YOUR TURN: X, y, y_err, the quality cut, the 80/20 split, standardization
X = np.asarray(t["coeffs"], dtype=np.float32)
Y = np.asarray(t["ALPHA_M"], dtype=np.float32)
y_err = np.asarray(t["ALPHA_M_ERR"], dtype=np.float32)
extras = np.column_stack((t["TEFF"], t["LOGG"], t["M_H"])).astype(np.float32)
mask = np.isfinite(Y) & np.isfinite(y_err) & (y_err > 0)
mask &= np.isfinite(extras).all(axis=1)
mask &= extras[:, 0] < 5000    # TEFF, column 0 in your stack order
mask &= extras[:, 1] < 3.5     # LOGG, the usual giant/dwarf divide
print("after finiteness:", mask.sum())

after finiteness: 280711


In [91]:
X = X[mask]
Y = Y[mask]
y_err = y_err[mask]
extras = extras[mask]

In [92]:
RNG = np.random.default_rng(2003)
idx = RNG.permutation(len(X))
ratio = int(0.8 * len(X))
train = idx[:ratio]
test = idx[ratio:]

In [93]:
train_X = X[train]
train_Y = Y[train]
train_y_err = y_err[train]
train_extras = extras[train]

In [94]:
test_X = X[test]
test_Y = Y[test]
test_y_err = y_err[test]
test_extras = extras[test]

In [97]:
norm_mu = train_X.mean(axis=0)     # shape (110,), one mean per coefficient
norm_sd = train_X.std(axis=0)      # shape (110,), one spread per coefficient

train_Xs = (train_X - norm_mu) / norm_sd
test_Xs  = (test_X  - norm_mu) / norm_sd

In [100]:
print(np.abs(train_Xs.mean(axis=0)).max(), train_Xs.std(axis=0).min(), train_Xs.std(axis=0).max())
print(np.abs(test_Xs.mean(axis=0)).max(),  test_Xs.std(axis=0).min(),  test_Xs.std(axis=0).max())

2.0433454e-05 0.99995357 1.0000945
0.010412545 0.62808704 1.1377065


In [101]:
j = test_Xs.std(axis=0).argmin()          # the offending coefficient
print(j, np.abs(train_Xs[:, j]).max(), np.abs(test_Xs[:, j]).max())

41 358.19272 26.865396


In [96]:
n = len(X)

# split bookkeeping: nothing lost, nothing double-counted, no leakage
print("rows:", n, "  train:", len(train_X), "  test:", len(test_X),
      "  sum ok:", len(train_X) + len(test_X) == n)
print("train/test overlap:", len(np.intersect1d(train, test)))   # must be 0

# usability: how many rows could actually enter a Gaussian likelihood
print("nan Y:", np.isnan(Y).sum())
print("nan y_err:", np.isnan(y_err).sum(), "  nonpositive y_err:", (y_err <= 0).sum())
print("bad X rows:", (~np.isfinite(X).all(axis=1)).sum())
print("bad extras rows:", (~np.isfinite(extras).all(axis=1)).sum())

# label sanity, nan-proof so it prints even before the cut
print("Y range:", np.nanmin(Y), "to", np.nanmax(Y), "  spread:", np.nanstd(Y))
print("median y_err:", np.nanmedian(y_err), " (must be well below the spread)")

rows: 280711   train: 224568   test: 56143   sum ok: True
train/test overlap: 0
nan Y: 0
nan y_err: 0   nonpositive y_err: 0
bad X rows: 0
bad extras rows: 0
Y range: -0.6597347 to 0.90503496   spread: 0.09937796
median y_err: 0.0069740512  (must be well below the spread)


after finiteness: 470041


## 1. What the paper did, and the baseline you have to beat

Your notes call this project "deep nearest neighbor". One correction before
you repeat it anywhere: the Laroche and Speagle paper never uses nearest
neighbors. What it actually does: train a variational autoencoder on the
110 coefficients alone, no labels in training, compressing each spectrum to
6 latent numbers. The known high-alpha and low-alpha sequences of cool
giants then separate visibly in that latent space, and a random forest fed
the 6 latents classifies high vs low alpha at 0.84 accuracy, while the same
classifier fed the stellar labels themselves (Teff, logg, [M/H]) reaches
0.70. The extra accuracy had to come from the spectra, and no label ever
entered the model. That is the "stellar label independent evidence" of the
title.

So "deep nearest neighbor" is Josh's name for what comes next, not the
paper's method. Plausibly it means neighbor-style prediction in a learned
representation. Confirm with him; it is on the section 8 question list.

Whatever the deep model ends up being, a regression project needs a floor.
The cheapest credible one: predict a star's [alpha/M] as the average of its
k nearest neighbors in coefficient space. A sampled deep net only earns its
cost if it beats this.

YOUR TURN. On the standardized coefficients, for each held-out star find
its k = 10 nearest training stars and predict the mean of their labels.
Compare against the global baseline, predicting the training mean for
everyone.

Hints: `torch.cdist` on the device does the distances; process the held-out
stars in chunks of about a thousand so memory stays flat. `torch.topk` with
`largest=False` gives the neighbor indices.

Checkpoint: report RMSE for both. The neighbor prediction should beat the
global mean clearly. Write the two numbers down, they are the bar the
sampled network has to clear in section 5.

In [ ]:
# YOUR TURN: k-NN prediction of [alpha/M] for the held-out stars, RMSE table


Optional, a label-independent check in the same spirit as the paper,
in miniature: take many random pairs of stars and many neighbor pairs, and
compare the spread of the label difference in the two sets. Neighbor pairs
agreeing more than random pairs is evidence the spectra carry the label,
and nothing was trained on the label to show it.

In [ ]:
# YOUR TURN (optional): neighbor pairs vs random pairs, two histograms


## 2. Regression instead of classification

Everything in exp6 carries over except one line: the likelihood. MNIST used
a categorical likelihood, the summed cross entropy. A real-valued label
wants a Gaussian:

$$-\ln p(y \mid x, \theta) = \frac{(y - \mu_\theta(x))^2}{2\sigma^2}
  + \ln \sigma + \tfrac{1}{2}\ln 2\pi$$

Two honest choices for $\sigma$, in increasing order of ambition:

1. use the APOGEE label error per star, $\sigma_i = $ `y_err[i]`. Simple and
   defensible: the model is asked to match each label to within its stated
   uncertainty.
2. add a learned intrinsic scatter $s$, so $\sigma_i^2 = $ `y_err[i]`$^2 +
   s^2$, with $s$ one extra sampled parameter. Do this later, if the
   calibration plot in section 5 says the label errors alone are too tight.

Start with choice 1. The misfit is again a sum of nats over the training
set, so the whole RTS machinery from exp6 applies unchanged.

YOUR TURN. Define the model and the misfit.

Model: an MLP from 110 inputs to 1 output. Start small, one or two hidden
layers of 64, grow it once the pipeline works end to end. Count the
parameters, D matters constantly in what follows.

Misfit: a function of the model returning the summed Gaussian negative log
likelihood over the training split, in nats, using `y_err` as sigma. Drop
the constant if you like, but then drop it everywhere.

Checkpoint: evaluate the misfit at random init and compare it to the misfit
of predicting the training mean for every star, computed with the same
formula. Random init should be much worse. Write both numbers down.

In [ ]:
# YOUR TURN: MLP(110 -> ... -> 1), parameter count D, misfit(model)


## 3. Priors: the actual lesson of exp6 (meeting note 1)

In exp6 the prior was N(0,1) on every weight and the whole story was
geometric: that prior's mass lives on the shell $\|\theta\|^2 \approx D$,
trained networks live nowhere near it, and the chains spent their entire
budget walking to the shell. Josh's note says: pick a prior whose shell sits
where sensible networks already live. Shrink the typical set.

The standard recipes are per layer. For a layer with fan_in inputs and
fan_out outputs:

    N(0, 1)              what exp6 used, one scale for everything
    Glorot               variance 2 / (fan_in + fan_out)
    fan-in (He style)    variance 1 / fan_in

A Glorot draw is what initialization theory says keeps signals order one as
they pass through the net. So the prior predictive, what the network does
before seeing any data, is sane by construction instead of saturated.

The 1/T note is the temperature dial from exp6: sample from
$\exp(\ln p / T)$. T = 1 is Bayes. Keep T = 1 until the calibration plot in
section 5 gives you a reason to move it, then you can say you moved it with
cause.

YOUR TURN, part 1. Build the per-parameter scale vector. Walk
`model.named_parameters()`, decide fan_in and fan_out per tensor (for a
`Linear` weight of shape `(out, in)`, fan_in is `in`; give biases a modest
fixed scale like 0.1, just be consistent and write your choice down), and
emit a flat vector `sigma` aligned with the flattened parameters. Then

$$\ln p(\theta) = -\tfrac{1}{2} \sum_j (\theta_j / \sigma_j)^2 + \text{const}$$

Do it for all three priors so they are one keyword apart.

Checkpoint: the prior's typical squared norm is $\sum_j \sigma_j^2$. Print
it for N(0,1), Glorot, and fan-in. N(0,1) gives D. The other two should be
far smaller. That number is where the walk of section 4 ends, so smaller is
not cosmetic.

In [ ]:
# YOUR TURN: sigma vectors for N(0,1) / Glorot / fan-in, log_prior(theta)


YOUR TURN, part 2, the figure that settles the argument: the prior
predictive check. Draw 20 networks from each prior (multiply a standard
normal draw by the sigma vector, load it into the model), run the held-out
coefficients through them, and histogram the predicted [alpha/M] under each
prior against the range of the real labels.

Checkpoint: under N(0,1) the predictions should be absurd, orders of
magnitude outside the label range. Under Glorot they should be order one.
If the panels look the same, your sigma vector is not actually being
applied. This single figure is the justification for meeting note 1, keep
it for Josh.

In [ ]:
# YOUR TURN: prior predictive histograms, one panel per prior


## 4. Sample it with RTS

Now wire the new target into the exp6 machinery. Read
`experiments/exp6_sample_metropolis.py` first: `make_log_prob` builds the
differentiable ln posterior from a flat parameter vector, and `run` hands it
to the vendor's `sample_raytrace`. Notice `make_log_prob` hard-codes the
N(0,1) prior as `-||theta||^2 / 2`. Your section 3 sigma vector generalizes
it: the prior term becomes `-0.5 * ((theta / sigma)**2).sum()`.

Plan, same as exp6: pilot short chains to tune dt for roughly 80 percent
acceptance, then one production chain per prior you care about, starting
with Glorot. Start every chain from a draw of its own prior. That is the
honest start and, with a Glorot prior, no longer an expensive one.

YOUR TURN, part 1. Write your own `make_log_prob(model, X, y, y_err,
sigma)` for the regression target, returning a function of the flat theta.
Test it before sampling: check that minus the log prob, minus the prior
term, equals your section 2 misfit at a few random thetas, and that
gradients flow (`torch.autograd.grad` returns finite numbers).

Checkpoint: the identity holds to float32 precision and the gradient norm
is finite at a prior draw.

In [ ]:
# YOUR TURN: make_log_prob for the regression posterior, plus the two tests


YOUR TURN, part 2. Pilot, then run. Sweep dt over a few values with
short chains (a couple hundred trajectories, L = 30 to start) and pick the
largest dt that keeps acceptance near 80 percent. Then run a production
chain, a few thousand trajectories, saving theta every trajectory like exp6
does. Apply the exp6 drift rule to the ln posterior trace to decide whether
it converged.

Checkpoint: acceptance in the 60 to 90 range, and the trace goes flat. A
prediction to test while you wait: exp6 needed 20,000 trajectories mostly
to walk to the N(0,1) shell. With the Glorot prior the start is already on
the shell, so convergence should be far faster. Confirm or refute with the
trace, either answer is a result for the meeting.

In [ ]:
# YOUR TURN: dt pilot, production chain, drift verdict


## 5. Visualize the uncertainty, compress the samples (meeting note 4)

You now have M posterior members. For regression the predictive per star is
a distribution over [alpha/M]: mean $\hat\mu_i$ across members, spread
$\hat\tau_i$ the member standard deviation, and total predicted uncertainty
$\hat\sigma_i^2 = \hat\tau_i^2 + $ `y_err[i]`$^2$.

Three plots tell you whether the posterior is doing its job, then one
exercise answers how few samples you actually need to keep.

YOUR TURN, part 1, accuracy: predicted vs true [alpha/M] on the held-out
stars, colored by $\hat\tau_i$. Put the section 1 k-NN RMSE and the chain
RMSE in the title.

Part 2, calibration, the plot that matters most: the z scores
$z_i = (y_i - \hat\mu_i) / \hat\sigma_i$ should be close to N(0,1) if the
uncertainties mean anything. Histogram z against a standard normal curve.
Too wide means underconfident, too narrow means overconfident, and this is
where a temperature or an intrinsic scatter s earns a justification.

Part 3, does uncertainty rank errors: sort held-out stars by $\hat\tau_i$,
flag the top 11 percent (the exp6 budget, keeps things comparable), and
report what fraction of the largest absolute errors land in the flagged
set.

Checkpoint: one sentence per plot on what it shows. If you cannot write the
sentence, the plot is not done.

In [ ]:
# YOUR TURN: predicted vs true


In [ ]:
# YOUR TURN: z-score calibration histogram


In [ ]:
# YOUR TURN: flag the top 11 percent by spread, report error capture


YOUR TURN, part 4, compression. Two independent senses of "fewer
samples", measure both:

1. how few members: recompute held-out RMSE and mean NLL using only the
   first k members, k = 1, 2, 5, 10, 20 and so on (the exp6 scaling curve,
   same code shape). The elbow is your answer to "how many nets do we
   ship".
2. how independent the members are: the autocorrelation time of the ln
   posterior trace says how far apart trajectories must be to count as
   fresh draws. Thin at that spacing and check that the plots above do not
   move.

Checkpoint: one number each, members needed and thinning stride.

In [ ]:
# YOUR TURN: scaling curve over member count, autocorrelation thinning


## 6. Look inside: layers and correlations (meeting note 5)

The posterior is a cloud in D dimensions and you hold M points of it. Two
cheap views reveal a lot.

Per layer: for each layer, compare the posterior standard deviation of its
weights across members to the prior sigma you gave that layer. A ratio near
1 means the data never touched those directions, the prior is doing all the
work there. A ratio well below 1 means the likelihood pinned them. This is
a map of where the data actually lives in the network.

Correlations: pick a manageable subset of weights, say 50 at random per
layer, and look at their correlation matrix across members. Structure in
the off-diagonal blocks tells you whether layers move together.

In [ ]:
# YOUR TURN: per-layer posterior std / prior sigma, one bar per layer


In [ ]:
# YOUR TURN: correlation matrix of a weight subset across members, imshow


## 7. GaiaXpy: look at the actual spectra (meeting note 3)

GaiaXpy is the official Gaia DPAC package for XP spectra. The archive (and
your table) stores each spectrum as basis-function coefficients; GaiaXpy's
`calibrate` turns them into an absolutely calibrated spectrum, flux against
wavelength in nm, so you can finally look at a star. It also does synthetic
photometry (`generate`) and plotting (`plot_spectra`).

`calibrate` accepts a pandas DataFrame with archive column names, so you
can feed it rows built from your arrays. Required: `source_id`,
`bp_n_parameters` and `rp_n_parameters` (both 55 for DR3),
`bp_standard_deviation` and `rp_standard_deviation`, plus the coefficient
blocks: `bp_coefficients`, `bp_coefficient_errors`,
`bp_coefficient_correlations` (the flattened triangle, 1485 values), and
the same three for rp. The pattern:

```python
import pandas as pd
from gaiaxpy import calibrate, plot_spectra

df = pd.DataFrame({
    "source_id": [sid],
    "bp_n_parameters": [55], "rp_n_parameters": [55],
    "bp_standard_deviation": [bp_std], "rp_standard_deviation": [rp_std],
    "bp_coefficients": [bp[:55]], "rp_coefficients": [rp[:55]],
    "bp_coefficient_errors": [bp_err], "rp_coefficient_errors": [rp_err],
    "bp_coefficient_correlations": [bp_corr_flat],
    "rp_coefficient_correlations": [rp_corr_flat],
})
calibrated, sampling = calibrate(df, save_file=False)
plot_spectra(calibrated, sampling=sampling, multi=True)
```

Gotchas, updated with what section 0 established about this table. Pass
`save_file=False` or it writes files. Plot the DataFrame that `calibrate`
returned, some pandas operations strip the metadata `plot_spectra` needs.
Feed RAW coefficients: the table's `coeffs` column is unnormalized (values
in the thousands), so build the DataFrame from the original arrays, not
from `train_Xs` / `test_Xs`; if only standardized arrays survive at this
point, undo with `Xs * norm_sd + norm_mu`. The errors exist:
`coeff_errs` is (N, 110) in the same layout as `coeffs`, so split it the
same way you split the coefficients. The correlations do not (they live in
`xp_corrs.npz` for a test subset only), so feed zeros there: the flux
curve depends only on the coefficients, the error band is then
approximate, and the figure caption should say so. If no per-source
standard-deviation column turns up, 1.0 is an acceptable placeholder for
`bp_standard_deviation` / `rp_standard_deviation` (it scales errors, not
flux). The 110-wide layout is assumed to be 55 BP then 55 RP; if the
calibrated spectra come out looking inside-out (blue flux where red should
be), swap the halves and note the finding.

YOUR TURN. Pick three held-out stars: lowest, median, and highest
[alpha/M]. Build the DataFrame, calibrate, and plot the three spectra on
one axis, labeled by their [alpha/M].

Checkpoint: the curves look like stellar spectra, smooth, positive, one
broad peak, and you can point at where they differ. Whether [alpha/M] is
visible by eye is genuinely interesting either way, note what you see.

In [ ]:
# YOUR TURN: three stars, calibrated spectra, one labeled figure


## 8. What to bring to the next meeting

The workbook is done when you can show, in order: the prior predictive
figure from section 3 (note 1 answered), the baseline table from sections 1
and 5, k-NN vs sampled net vs global mean (note 2 answered), the z-score
calibration plot with your T = 1 verdict, the members-needed and thinning
numbers (note 4 answered), the layer std-ratio bar chart (note 5 answered),
and one GaiaXpy spectrum figure (note 3 answered).

Questions to raise when you present: what exactly did you mean by deep
nearest neighbor (the paper itself uses a VAE latent space, not neighbors);
should sigma include a learned intrinsic scatter; is [alpha/M] alone the
right target or should the net predict several labels at once; and what is
the compression target, fewer members or something smaller than full weight
vectors.